In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_absolute_error

In [3]:
# Dosya yolu
df = pd.read_csv("all_data_ratio.csv")

In [4]:
# Feature map
feature_map = {
    "uzman": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "diger"],
    "pratisyen": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "ebe", "diger"],
    "asistan": ["yıl", "nufus", "hastane", "yatak", "uzman", "pratisyen", "hemsire", "ebe", "diger"],
    "ebe": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "pratisyen", "diger"],
    "diger": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "uzman"],
    "hemsire": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "yatak": ["yıl", "nufus", "hastane", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "hastane": ["yıl", "nufus", "yatak", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"]
}

In [8]:
tahminler = []
tscv = TimeSeriesSplit(n_splits=2)  # Daha güvenli split
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

for il in df["il_adı"].unique():
    df_il = df[df["il_adı"] == il]
    if df_il[df_il["yıl"] <= 2021].shape[0] < 5:
        continue

    for hedef, features in feature_map.items():
        if hedef not in df_il.columns or any(f not in df_il.columns for f in features):
            continue

        df_target = df_il[features + [hedef]].dropna()
        if df_target.empty or df_target[df_target["yıl"] <= 2021].shape[0] == 0:
            continue

        # Train: 2002-2021, Validation: 2022, Test: 2023
        train_data = df_target[df_target['yıl'] <= 2021]
        val_data = df_target[df_target['yıl'] == 2022]
        test_data = df_target[df_target['yıl'] == 2023]

        if val_data.empty or test_data.empty:
            continue

        X_train = train_data[features]
        y_train = train_data[hedef]
        X_val = val_data[features]
        y_val = val_data[hedef]
        X_test = test_data[features]
        y_test = test_data[hedef]

        # Küçük veya sabit veri seti kontrolü
        if len(X_train) < 10 or len(set(y_train)) <= 1:
            continue

        # Linear Regression (sabit)
        lr = LinearRegression().fit(X_train, y_train)

        # Random Forest - GridSearchCV (korumalı)
        try:
            rf_params = {'n_estimators': [50, 100], 'max_depth': [None, 5]}
            rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=tscv,
                                   n_jobs=-1, scoring=mae_scorer)
            rf_grid.fit(X_train, y_train)
            rf = rf_grid.best_estimator_
        except:
            rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

        # XGBoost (sabit)
        xgb = XGBRegressor(objective='reg:squarederror', random_state=42).fit(X_train, y_train)

        # SVR - RandomizedSearchCV (korumalı)
        try:
            svr_params = {'C': [1, 10], 'epsilon': [0.1]}
            svr_rand = RandomizedSearchCV(SVR(), svr_params, cv=tscv, n_iter=2, n_jobs=-1,
                                          random_state=42, scoring=mae_scorer)
            svr_rand.fit(X_train, y_train)
            svr = svr_rand.best_estimator_
        except:
            svr = SVR(C=1.0, epsilon=0.1).fit(X_train, y_train)

        # KNN - GridSearchCV (korumalı, dinamik komşu ayarı)
        try:
            max_neighbors = min(7, len(X_train))
            knn_k_list = [k for k in [3, 5, 7] if k <= max_neighbors]

            if knn_k_list:
                knn_params = {'n_neighbors': knn_k_list}
                knn_grid = GridSearchCV(KNeighborsRegressor(), knn_params, cv=tscv,
                                        n_jobs=-1, scoring=mae_scorer)
                knn_grid.fit(X_train, y_train)
                knn = knn_grid.best_estimator_
            else:
                knn = KNeighborsRegressor(n_neighbors=1).fit(X_train, y_train)
        except:
            knn = KNeighborsRegressor(n_neighbors=1).fit(X_train, y_train)

                # XGBoost + RandomizedSearchCV (genis parametre araligi)
        xgb_params = {
            "n_estimators": [100, 200],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.01, 0.05, 0.1],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
            "min_child_weight": [1, 3],
            "gamma": [0, 0.1, 0.2],
            "reg_alpha": [0, 0.1],
            "reg_lambda": [1, 2]
        }
        
        try:
            grid = RandomizedSearchCV(
                estimator=XGBRegressor(objective='reg:squarederror', random_state=42),
                param_distributions=xgb_params,
                scoring=mae_scorer,
                n_iter=50,
                cv=tscv,  # Hatalıysa düzeltildi
                n_jobs=-1,
                verbose=0,
                random_state=42
            )
            grid.fit(X_train, y_train)
            xgb = grid.best_estimator_
            best_params = grid.best_params_  # ✅ parametreleri kaydet
        except:
            xgb = XGBRegressor(objective='reg:squarederror', random_state=42).fit(X_train, y_train)
            best_params = {}  # Boş bırak

        # Tahmin ve skorlar
        y_val_pred_lr = lr.predict(X_val)[0]
        y_val_pred_rf = rf.predict(X_val)[0]
        y_val_pred_xgb = xgb.predict(X_val)[0]
        y_val_pred_svr = svr.predict(X_val)[0]
        y_val_pred_knn = knn.predict(X_val)[0]

        y_test_pred_lr = lr.predict(X_test)[0]
        y_test_pred_rf = rf.predict(X_test)[0]
        y_test_pred_xgb = xgb.predict(X_test)[0]
        y_test_pred_svr = svr.predict(X_test)[0]
        y_test_pred_knn = knn.predict(X_test)[0]

        tahminler.append({
            "il_adı": il,
            "hedef": hedef,
            "gercek_2022": y_val.values[0],
            "tahmin_2022_lr": y_val_pred_lr,
            "tahmin_2022_rf": y_val_pred_rf,
            "tahmin_2022_xgb": y_val_pred_xgb,
            "tahmin_2022_svr": y_val_pred_svr,
            "tahmin_2022_knn": y_val_pred_knn,
            "gercek_2023": y_test.values[0],
            "tahmin_2023_lr": y_test_pred_lr,
            "tahmin_2023_rf": y_test_pred_rf,
            "tahmin_2023_xgb": y_test_pred_xgb,
            "_best_params": best_params,
            "tahmin_2023_svr": y_test_pred_svr,
            "tahmin_2023_knn": y_test_pred_knn,
        })


In [10]:
# XGBoost en iyi parametreleri listele ve ekrana yazdır
xgb_best_params = []

for result in tahminler:
    if f"tahmin_2023_xgb" in result:
        try:
            best_params = result.get("_best_params", {})
            if best_params:
                print(f"\n📍 İl: {result['il_adı']} | Hedef: {result['hedef']}")
                for param, val in best_params.items():
                    print(f"  - {param}: {val}")

                xgb_best_params.append({
                    "il_adı": result["il_adı"],
                    "hedef": result["hedef"],
                    **best_params
                })
        except:
            continue

# CSV'ye de kaydet (opsiyonel)
df_xgb_params = pd.DataFrame(xgb_best_params)

if not df_xgb_params.empty:
    print("\n📋 İlk 5 XGBoost parametresi:")
    print(df_xgb_params.head())

    df_xgb_params.to_csv("xgb_secilen_parametreler.csv", index=False)
    print("\n✅ XGBoost parametreleri 'xgb_secilen_parametreler.csv' olarak kaydedildi.")
else:
    print("⚠️ Herhangi bir parametre bulunamadı, muhtemelen grid search hiç başarılı çalışmadı.")

⚠️ Herhangi bir parametre bulunamadı, muhtemelen grid search hiç başarılı çalışmadı.


In [112]:
df_tahmin = pd.DataFrame(tahminler)

In [12]:
# Doğruluk oranı hesaplama (2023 yılı için)
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    tahmin_kolonu = f"tahmin_2023_{model}"
    df_tahmin[f"{model}_yuzde_dog"] = 100 - (
        (np.abs(df_tahmin["gercek_2023"] - df_tahmin[tahmin_kolonu]) / df_tahmin["gercek_2023"]) * 100
    )
    # Değerleri 0–100 aralığına kısıtla
    df_tahmin[f"{model}_yuzde_dog"] = df_tahmin[f"{model}_yuzde_dog"].clip(lower=0, upper=100)
print("\n📊 Şehir Bazlı Ortalama Doğruluklar (2023):")
print("-" * 40)
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ortalama = df_tahmin[f"{model}_yuzde_dog"].mean()
    print(f" {model.upper():<4} doğruluk ort.: {ortalama:.2f}%")
# Tahmin detayları içeren DataFrame'i CSV'ye yaz
df_tahmin.to_csv("sehir_bazli_2023_tahmin_dogruluk_5model.csv", index=False)
print("\n✅ Tahmin doğrulukları 'sehir_bazli_2023_tahmin_dogruluk_5model.csv' olarak kaydedildi.")


📊 Şehir Bazlı Ortalama Doğruluklar (2023):
----------------------------------------
 LR   doğruluk ort.: 67.85%
 RF   doğruluk ort.: 78.87%
 XGB  doğruluk ort.: 79.89%
 SVR  doğruluk ort.: 70.43%
 KNN  doğruluk ort.: 79.18%

✅ Tahmin doğrulukları 'sehir_bazli_2023_tahmin_dogruluk_5model.csv' olarak kaydedildi.


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("\n📉 Model Performans Metrikleri (2023):")
print("{:<6} | {:>10} | {:>10}".format("Model", "MAE", "RMSE"))
print("-" * 32)

metric_sonuclar = []

for model in ["lr", "rf", "xgb", "svr", "knn"]:
    y_true = df_tahmin["gercek_2023"]
    y_pred = df_tahmin[f"tahmin_2023_{model}"]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)

    print("{:<6} | {:>10.2f} | {:>10.2f}".format(model.upper(), mae, rmse))

    metric_sonuclar.append({
        "Model": model.upper(),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2)
    })
# Metrikleri CSV olarak kaydet
df_metrics = pd.DataFrame(metric_sonuclar)
df_metrics.to_csv("sehir_bazli_2023_model_metrikleri.csv", index=False)
print("\n✅ Performans metrikleri 'sehir_bazli_2023_model_metrikleri.csv' dosyasına kaydedildi.")


📉 Model Performans Metrikleri (2023):
Model  |        MAE |       RMSE
--------------------------------
LR     |     385.10 |    1676.15
RF     |     297.69 |    1052.59
XGB    |     248.50 |     880.46
SVR    |     524.30 |    1938.14
KNN    |     296.30 |     982.87

✅ Performans metrikleri 'sehir_bazli_2023_model_metrikleri.csv' dosyasına kaydedildi.


C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared err

In [16]:
# Türkiye'nin 7 coğrafi bölgesi
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

def bolge_bul(il):
    for bolge, iller in bolge_haritasi.items():
        if il in iller:
            return bolge
    return "Bilinmiyor"

# Bölge ekle
df_tahmin["bolge"] = df_tahmin["il_adı"].apply(bolge_bul)

# Bölgelere göre grupla ve toplamları al
bolgesel = df_tahmin.groupby(["bolge", "hedef"])[[
    "gercek_2023",
    "tahmin_2023_lr",
    "tahmin_2023_rf",
    "tahmin_2023_xgb",
    "tahmin_2023_svr",
    "tahmin_2023_knn"
]].sum().reset_index()

# CSV olarak kaydet
bolgesel.to_csv("bolge_bazli_2023_tahmin5model.csv", index=False)

In [18]:
# 1. Dosyayı oku
df = pd.read_csv("sehir_bazli_2023_tahmin_dogruluk_5model.csv")  # Gerekirse yolunu düzenle

# 2. Doğruluk hesaplama fonksiyonu
def hesapla_dog(model_etiketi):
    df[f"{model_etiketi}_yuzde_dog"] = 100 - (
        np.abs(df["gercek_2023"] - df[f"tahmin_2023_{model_etiketi}"]) / df["gercek_2023"] * 100
    )
    df[f"{model_etiketi}_yuzde_dog"] = df[f"{model_etiketi}_yuzde_dog"].clip(0, 100)

# 3. Altı model için hesapla
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    hesapla_dog(model)

# 4. Ortalamaları yazdır
print("\nŞehir Bazlı Doğruluk Ortalamaları (2023):")
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ortalama = round(df[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ortalama}%")

# 5. Güncel dosya olarak kaydet
df.to_csv("sehir_tahmin_yuzdeler.csv", index=False)


Şehir Bazlı Doğruluk Ortalamaları (2023):
 - LR: 67.85%
 - RF: 78.87%
 - XGB: 79.89%
 - SVR: 70.43%
 - KNN: 79.18%


In [20]:
# 1. Dosyayı oku
df = pd.read_csv("bolge_bazli_2023_tahmin5model.csv")  # Gerekirse dosya adını değiştir

# 2. Her model için yüzdelik doğruluk hesapla
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    df[f"{model}_yuzde_dog"] = 100 - (
        np.abs(df["gercek_2023"] - df[f"tahmin_2023_{model}"]) / df["gercek_2023"] * 100
    )
    df[f"{model}_yuzde_dog"] = df[f"{model}_yuzde_dog"].clip(0, 100)

# 3. Ortalamaları hesapla ve yazdır
print(f"\nBölge Doğruluk Ortalamaları (2023):")
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ortalama = round(df[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ortalama}%")

# 4. Dosyaya kaydet
df.to_csv("bolge_tahmin_yuzdeler.csv", index=False)


Bölge Doğruluk Ortalamaları (2023):
 - LR: 83.93%
 - RF: 84.88%
 - XGB: 87.42%
 - SVR: 73.97%
 - KNN: 84.89%


In [75]:
import warnings
warnings.filterwarnings("ignore")

# 1. Tahmin çıktısını oku
df = pd.read_csv("sehir_bazli_2023_tahmin_dogruluk_5model.csv")

# 2. En iyi 3 modeli doğruluk ortalamasına göre belirle
model_list = ["lr", "rf", "xgb", "svr", "knn"]
dog_list = [(model, df[f"{model}_yuzde_dog"].mean()) for model in model_list]
dog_list_sorted = sorted(dog_list, key=lambda x: x[1], reverse=True)

# İlk 3 modeli seç
en_iyi_uc = [m[0] for m in dog_list_sorted[:3]]
print("\n✅ En iyi 3 model:", en_iyi_uc)

# 3. Ağırlık kombinasyonlarını oluştur
grid = []
for w1 in np.arange(0.1, 1.0, 0.1):
    for w2 in np.arange(0.1, 1.0 - w1, 0.1):
        w3 = 1.0 - w1 - w2
        if w3 > 0:
            grid.append((round(w1, 2), round(w2, 2), round(w3, 2)))

en_iyi_agirlik = None
en_iyi_ortalama = 0

# 4. Her ağırlık kombinasyonu için doğruluk hesapla
for w in grid:
    ensemble_tahmin = (
        df[f"tahmin_2023_{en_iyi_uc[0]}"] * w[0] +
        df[f"tahmin_2023_{en_iyi_uc[1]}"] * w[1] +
        df[f"tahmin_2023_{en_iyi_uc[2]}"] * w[2]
    )
    ensemble_dog = 100 - (np.abs(df["gercek_2023"] - ensemble_tahmin) / df["gercek_2023"] * 100)
    ensemble_dog = ensemble_dog.clip(0, 100)
    ort = ensemble_dog.mean()

    if ort > en_iyi_ortalama:
        en_iyi_ortalama = ort
        en_iyi_agirlik = w
        df["tahmin_2023_ensemble"] = ensemble_tahmin
        df["ensemble_yuzde_dog"] = ensemble_dog

# 5. Sonuçları yazdır
print(f"\n🎯 En iyi ağırlık kombinasyonu: {en_iyi_agirlik}")
print("📊 Ensemble doğruluk ortalaması (2023): {:.2f}%".format(en_iyi_ortalama))

# 6. Kaydet
df.to_csv("sehir_bazli_2023_ensemble_model.csv", index=False)
print("💾 Ensemble çıktısı 'sehir_bazli_2023_ensemble_model.csv' olarak kaydedildi.")



✅ En iyi 3 model: ['xgb', 'knn', 'rf']

🎯 En iyi ağırlık kombinasyonu: (0.7, 0.2, 0.1)
📊 Ensemble doğruluk ortalaması (2023): 80.02%
💾 Ensemble çıktısı 'sehir_bazli_2023_ensemble_model.csv' olarak kaydedildi.


In [77]:
# 1. Dosyayı oku
df = pd.read_csv("sehir_tahmin_yuzdeler.csv")

# 2. En iyi 3 modelin doğruluk ortalamalarına göre seçimi
modeller = ["lr", "rf", "xgb", "svr", "knn"]
ortalamalar = {model: df[f"{model}_yuzde_dog"].mean() for model in modeller}
en_iyi3 = sorted(ortalamalar.items(), key=lambda x: x[1], reverse=True)[:3]

mod_adlari = [m[0] for m in en_iyi3]
print(f"\n🔍 Seçilen modeller: {mod_adlari}")

# 3. Sabit ağırlıkları tanımla
# 3. Ağırlık kombinasyonlarını tanımla (örnek: 0.6, 0.3, 0.1 gibi)
grid = []
for w1 in np.arange(0.1, 1.0, 0.1):
    for w2 in np.arange(0.1, 1.0 - w1, 0.1):
        w3 = 1.0 - w1 - w2
        if w3 > 0:
            grid.append((round(w1, 2), round(w2, 2), round(w3, 2)))
en_iyi_agirlik = None
en_iyi_ortalama = 0

# 4. Ensemble tahmini oluştur
df["tahmin_2023_ensemble"] = (
    df[f"tahmin_2023_{mod_adlari[0]}"] * agirliklar[0] +
    df[f"tahmin_2023_{mod_adlari[1]}"] * agirliklar[1] +
    df[f"tahmin_2023_{mod_adlari[2]}"] * agirliklar[2]
)

# 5. Ensemble doğruluğu hesapla
df["ensemble_yuzde_dog"] = 100 - (
    np.abs(df["gercek_2023"] - df["tahmin_2023_ensemble"]) / df["gercek_2023"] * 100
)
df["ensemble_yuzde_dog"] = df["ensemble_yuzde_dog"].clip(0, 100)

# 6. Ortalamayı yazdır
ortalama = df["ensemble_yuzde_dog"].mean()
print(f"\n📊 Sabit Ağırlıklı Ensemble Doğruluk Ortalaması: {ortalama:.2f}%")

# 7. CSV kaydı
df.to_csv("sehir_tahmin_yuzdeler_ensemble.csv", index=False)
print("💾 Dosya 'sehir_tahmin_yuzdeler_ensemble.csv' olarak kaydedildi.")



🔍 Seçilen modeller: ['xgb', 'knn', 'rf']

📊 Sabit Ağırlıklı Ensemble Doğruluk Ortalaması: 80.02%
💾 Dosya 'sehir_tahmin_yuzdeler_ensemble.csv' olarak kaydedildi.


In [79]:
# 1. Dosyayı oku
df = pd.read_csv("sehir_tahmin_yuzdeler_ensemble.csv")  # ensemble doğruluk içeren dosya

# 2. Bölge haritası
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

def bolge_bul(il):
    for bolge, iller in bolge_haritasi.items():
        if il in iller:
            return bolge
    return "Bilinmiyor"

# 3. Bölge bilgisi ekle
df["bolge"] = df["il_adı"].apply(bolge_bul)

# 4. Bölge ve hedef bazında grupla, tahmin ve gerçekleri topla
bolge_df = df.groupby(["bolge", "hedef"])[["gercek_2023", "tahmin_2023_ensemble"]].sum().reset_index()

# 5. Ensemble doğruluğunu hesapla
bolge_df["ensemble_yuzde_dog"] = 100 - (
    np.abs(bolge_df["gercek_2023"] - bolge_df["tahmin_2023_ensemble"]) / bolge_df["gercek_2023"] * 100
)
bolge_df["ensemble_yuzde_dog"] = bolge_df["ensemble_yuzde_dog"].clip(0, 100)

# 6. Ortalama doğruluğu yazdır
ortalama = round(bolge_df["ensemble_yuzde_dog"].mean(), 2)
print(f"\n📍 Ensemble Bölge Doğruluk Ortalaması: {ortalama}%")

# 7. CSV olarak kaydet
bolge_df.to_csv("bolge_tahmin_yuzdeler_ensemble.csv", index=False)
print("💾 'bolge_tahmin_yuzdeler_ensemble.csv' dosyası kaydedildi.")



📍 Ensemble Bölge Doğruluk Ortalaması: 86.66%
💾 'bolge_tahmin_yuzdeler_ensemble.csv' dosyası kaydedildi.


ValueError: Cannot describe a DataFrame without columns